In [0]:
%run "../config/paths"

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

In [0]:
df = spark.read.table("bronze.readmission")

In [0]:
  
df = df.na.replace(["NA", "N/A", ""], None)

In [0]:
performance_year_df = df.withColumn("performance_year", year(to_date(col("end_date"), 'MM/dd/yyyy')))

In [0]:
is_excessive_flag_df = performance_year_df.withColumn("is_excessive_flag", 
                                    when(col("excess_readmission_ratio").cast('float') > 1, True)
                                    .when(col("excess_readmission_ratio").cast('float') < 1, False)
                                    .otherwise(None)
                                )

In [0]:
condition_code_df = is_excessive_flag_df.withColumn("condition_code",
        regexp_extract(col("measure_name"), r"READM-\d+-(.*)-HRRP", 1)     
    )

In [0]:
condition_mapping = {
        "AMI": "Acute Myocardial Infarction",
        "COPD": "Chronic Obstructive Pulmonary Disease",
        "HF": "Heart Failure",
        "PN": "Pneumonia",
        "CABG": "Coronary Artery Bypass Graft Surgery",
        "THA/TKA": "Total Hip and/or Total Knee Arthroplasty",
        "HIP-KNEE": "Total Hip and/or Total Knee Arthroplasty"
}

mapping_df = spark.createDataFrame(
    [(k, v) for k, v in condition_mapping.items()],
    ["condition_code", "condition_name"]
)

condition_name_df = condition_code_df.join(broadcast(mapping_df),
        on="condition_code",
        how="left"
)

In [0]:
std_cols_df = (condition_name_df
    .withColumn("facility_id", upper(col("facility_id")))
    .withColumn("state", upper(col("state"))).withColumnRenamed("state", "state_code")
    .withColumn("measure_name", upper(col("measure_name")))
    .withColumn("condition_code", upper(col("condition_code")))
    .withColumn("facility_name", initcap(col("facility_name")))
    .withColumn("condition_name", initcap(col("condition_name")))
    .withColumn("footnote", lower(col("footnote")))
)

In [0]:
##hash key

hash_df = (
        std_cols_df.withColumn
                ("readmission_bk_hash", 
                        md5(concat_ws("|",
                            col("facility_id"),
                            col("condition_code"),
                            col("start_date"),
                            col("end_date") 
                    )
                )
            )
        )

In [0]:
window_spec = Window.partitionBy("readmission_bk_hash").orderBy(desc("load_timestamp"))

dedup_df = (
    hash_df
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num") # drop the col once the you get the deisred result
)

In [0]:
dedup_df = dedup_df.na.replace("Too Few to Report", None)

In [0]:
final_df = ( 
            dedup_df
            .withColumn("source_data", lit("bronze.readmission"))
            .withColumn("load_timestamp", current_timestamp())

            .withColumn("start_date", to_date(col("start_date"),'MM/dd/yyyy'))
            .withColumn("end_date", to_date(col("end_date"),'MM/dd/yyyy'))

            .withColumn("number_of_discharges", col("number_of_discharges").cast("int"))
            .withColumn("number_of_readmissions", col("number_of_readmissions").cast("int"))
            .withColumn("performance_year", col("performance_year").cast("int"))

            .withColumn("excess_readmission_ratio", col("excess_readmission_ratio").cast("float"))
            .withColumn("predicted_readmission_rate", col("predicted_readmission_rate").cast("float"))
            .withColumn("expected_readmission_rate", col("expected_readmission_rate").cast("float"))

            .withColumn("is_excessive_flag", col("is_excessive_flag").cast("boolean"))

            .withColumnRenamed("is_excessive_flag", "is_excess_readmission")
            .drop("rescued_data")
)

In [0]:
final_df.write.format("delta").mode("append").saveAsTable("silver.readmission")